# Chance-constrained programming: a portfolio optimization example

In [1]:
using LinearAlgebra
using JuMP
using HiGHS
using Distributions
using RandomDataStreams

In [2]:
using Clarabel

*Adapted from Stephen Boyd and Lieven Vandenberghe, "Convex Optimization", Cambridge University Press, 2004, Section 4.7.6, p. 187.*

We consider an investiment portfolio with $n$ assets and random returns, except the last one. We assume that the $n-1$ first returns follow a multivariate distribution with known mean and covariance matrix. The last asset is a risk-free investment product, with a fixed return.

The assets characteristics are detailed below.

In [3]:
n = 4   # number of assets

# The last asset is a risk-free asset, with a null variance.
# We ignore it in the covariance matrix in order to keep it positive definite.
μ = [.12 ; .10 ; .07 ; .03]
Σ = [ 4e-2  6e-3 -4e-3  ;
      6e-3  1e-2  0.0  ;
      -4e-3  0.0 2.5e-3 ]

# We create une multivariate normal of mean μ and covariance matrix Σ
d = MvNormal(μ[1:n-1], Σ)

FullNormal(
dim: 3
μ: [0.12, 0.1, 0.07]
Σ: [0.04 0.006 -0.004; 0.006 0.01 0.0; -0.004 0.0 0.0025]
)


The following function estimates by simulation the probability to have a negative return and the resulting expected shortfall.

The returns are drawn from a stream of the `RandomDataStreams` library, rewound at the beginning of each call. Every portfolio is therefore evaluated on the same scenarios (common random numbers), so that the differences we observe below come from the portfolios themselves and not from the sampling noise.

In [4]:
shortfall_gen = Xoshiro256ppGen(2024)
shortfall_rng = next_stream!(shortfall_gen)

function expectedshortfall(p:: Vector, d:: Distribution, M:: Int = 1000000;
                           rng:: AbstractStreamableRNG = shortfall_rng)

    reset_stream!(rng)  # common random numbers: same scenarios for every portfolio

    loss = 0   # frequency of losses
    vloss = 0  # value of the losses
    for i = 1:M
        ξ = [rand(rng, d); μ[n]]
        ret = dot(p, ξ)
        if ret < 0
            loss += 1
            vloss += ret
        end
    end

    return loss/M, vloss/loss
    
end

expectedshortfall (generic function with 2 methods)

## Portfolio with uniform repartition

We first consider the naive strategy where the same amount is invested in each asset.

In [5]:
# Expected return with a uniform repartition.
p = ones(n)./n
er = sum(p[i]*μ[i] for i = 1:n)

0.08000000000000002

We now compute the loss probability and the average loss when a loss occurs.

In [6]:
ploss, eshortfall = expectedshortfall(p, d)
println("Loss probability: ", ploss)
println("Expected shortfall: ", eshortfall)

Loss probability: 0.089134
Expected shortfall: -0.02743451891052764


## Optimal decision without loss constraint

We now aim to maximize the expected return, without any consideration for the potential loss. We allow to sell some assets in favor of other ones.

In [7]:
m = Model(HiGHS.Optimizer)

@variable(m, p[1:n] >= -0.1)
@constraint(m, sum(p[i] for i = 1:n) <= 1)

@objective(m, Max, sum(p[i]*μ[i] for i = 1:n))

println(m)

Max 0.12 p[1] + 0.1 p[2] + 0.07 p[3] + 0.03 p[4]
Subject to
 p[1] + p[2] + p[3] + p[4] ≤ 1
 p[1] ≥ -0.1
 p[2] ≥ -0.1
 p[3] ≥ -0.1
 p[4] ≥ -0.1



In [8]:
optimize!(m)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 1 row; 4 cols; 4 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [3e-02, 1e-01]
  Bound   [1e-01, 1e-01]
  RHS     [1e+00, 1e+00]
Presolving model
1 rows, 1 cols, 1 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-1); columns 0(-4); nonzeros 0(-4) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  1.3600000000e-01
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [9]:
println("Loss probability: ", value.(p))

Loss probability: [1.3, -0.1, -0.1, -0.1]


Not surprisingly, we invest everything is the asset having the highest return rate, even if it means borrowing from the lower return rate asset.

In [10]:
println("Expected return: ", objective_value(m))

Expected return: 0.13599999999999998


The loss probability is however close to 30%, and the average lost amount is significantly more important if a loss occurs.

In [11]:
ploss, eshortfall = expectedshortfall(value.(p), d)
println("Loss probability: ", ploss)
println("Expected shortfall: ", eshortfall)

Loss probability: 0.299567
Expected shortfall: -0.16437431720266193


## Optimal decision with loss constraint

We add the constraint that we want to limit the risk by accepting a loss with a maximum probability of 0.05.

In [12]:
maxloss = 0 # maximum tolerated loss

α = 0.95
z = 1/quantile(Normal(0,1), α)

0.6079568319117693

We build the second-order cone constraint corresponding to the joint chance constraint.

HiGHS does not handle second-order cone constraints. We switch to Clarabel, an open-source
interior-point solver for conic programs, and then add the constraint.

In [13]:
set_optimizer(m, Clarabel.Optimizer)

# || x || <= t, t >= 0
# https://jump.dev/JuMP.jl/stable/reference/constraints/#JuMP.SecondOrderCone
@constraint(m, [z*(-maxloss+sum(μ[i]*p[i] for i = 1:n)); (Σ^0.5)*p[1:n-1]] in SecondOrderCone())

println("Model:\n", m)

Model:
Max 0.12 p[1] + 0.1 p[2] + 0.07 p[3] + 0.03 p[4]
Subject to
 p[1] + p[2] + p[3] + p[4] ≤ 1
 [0.0729548198294123 p[1] + 0.06079568319117693 p[2] + 0.04255697823382386 p[3] + 0.018238704957353077 p[4], 0.19827331129330036 p[1] + 0.02038945638178348 p[2] - 0.016491334004420634 p[3], 0.02038945638178348 p[1] + 0.09787182302877327 p[2] + 0.002318690466585327 p[3], -0.016491334004420634 p[1] + 0.002318690466585327 p[2] + 0.04714509070173485 p[3]] ∈ MathOptInterface.SecondOrderCone(4)
 p[1] ≥ -0.1
 p[2] ≥ -0.1
 p[3] ≥ -0.1
 p[4] ≥ -0.1



We now optimize it.

In [14]:
optimize!(m)

-------------------------------------------------------------
           Clarabel.jl v0.11.1  -  Clever Acronym              

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 4
  constraints   = 9
  nnz(P)        = 0
  nnz(A)        = 21
  cones (total) = 2
    : Nonnegative = 1,  numel = 5
    : SecondOrder = 1,  numel = 4

settings:
  linear algebra: direct / qdldl, precision: 64 bit (1 thread)
  max iter = 200, time limit = Inf,  max step = 0.990
  tol_feas = 1.0e-08, tol_gap_abs = 1.0e-08, tol_gap_rel = 1.0e-08,
  static reg : on, ϵ1 = 1.0e-08, ϵ2 = 4.9e-32
  dynamic reg: on, ϵ = 1.0e-13, δ = 2.0e-07
  iter refine: on, reltol = 1.0e-13, abstol = 1.0e-12, 
               max iter = 10, stop ratio = 5.0
  equilibrate: on, min_scale = 1.0e-04, max_scale = 1.0e+04
               max iter = 10

iter    pcost        dcost   

In [15]:
println("Optimal portfolio: ", value.(p))
println("Expected return: ", objective_value(m))

Optimal portfolio: [0.18194024440622317, 0.3401479312181258, 0.5779117933906507, -0.09999998782770478]
Expected return: 0.09330144835307377


Without any surprise, the expected return is less than without the loss constraint, but is still higher than with the uniform repartition. We can also see that we use the risk-free asset as leverage to buy riskier assets. The risk to lose money is limited to 5%, as desired, and is less than any other strategy. Interestingly, the associated average loss is also the smallest one.

In [16]:
ploss, eshortfall = expectedshortfall(value.(p), d)
println("Loss probability: ", ploss)
println("Expected shortfall: ", eshortfall)

Loss probability: 0.049908
Expected shortfall: -0.023689519928156436
